<a href="https://colab.research.google.com/github/kasturikirankumar1101-lab/AI_TOOLS/blob/main/RAG_Injetion_with_Pinecone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-community langchain-openai langchain-text-splitters langchain-chroma

In [ ]:
!pip install -U pinecone-client langchain langchain-pinecone

In [ ]:
# =========================================================
# INSTALL REQUIRED PACKAGES (run once)
# =========================================================
# !pip install langchain langchain-openai langchain-community \
# langchain-text-splitters langchain-pinecone pinecone python-dotenv unstructured

# =========================================================
# IMPORTS
# =========================================================
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import (
    UnstructuredWordDocumentLoader,
    DirectoryLoader
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings

from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone
from google.colab import userdata


# =========================================================
# LOAD ENVIRONMENT VARIABLES
# =========================================================
load_dotenv()

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["PINECONE_API_KEY"] = userdata.get("PINECONE_API_KEY")

if not os.getenv("OPENAI_API_KEY") or not os.getenv("PINECONE_API_KEY"):
    raise ValueError("Missing API keys. Set OPENAI_API_KEY and PINECONE_API_KEY")


# =========================================================
# STEP 1: LOAD DOCUMENTS
# =========================================================
def load_documents(docs_path: str):
    """
    Load all .docx documents from a directory.

    Parameters
    ----------
    docs_path : str
        Path to the folder containing .docx files.

    Returns
    -------
    List[Document]
        Loaded LangChain Document objects.

    Raises
    ------
    FileNotFoundError
        If directory does not exist or contains no documents.
    """

    print(f"📂 Loading documents from: {docs_path}")

    if not os.path.exists(docs_path):
        raise FileNotFoundError(f"Directory not found: {docs_path}")

    loader = DirectoryLoader(
        path=docs_path,
        glob="*.docx",
        loader_cls=UnstructuredWordDocumentLoader
    )

    documents = loader.load()

    if not documents:
        raise FileNotFoundError("No .docx files found in directory")

    print(f"✅ Loaded {len(documents)} documents")

    # Preview first 2 docs
    for i, doc in enumerate(documents[:2]):
        print(f"\n📄 Document {i+1}")
        print(f"Source: {doc.metadata.get('source')}")
        print(f"Length: {len(doc.page_content)} chars")
        print(f"Preview: {doc.page_content[:150]}...")

    return documents


# =========================================================
# STEP 2: SPLIT DOCUMENTS INTO CHUNKS
# =========================================================
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """
    Split documents into smaller overlapping chunks.

    Why?
    ----
    LLMs and embeddings perform better on smaller chunks.

    Parameters
    ----------
    documents : List[Document]
    chunk_size : int
    chunk_overlap : int

    Returns
    -------
    List[Document]
    """

    print("✂️ Splitting documents into chunks...")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunks = splitter.split_documents(documents)

    print(f"✅ Created {len(chunks)} chunks")

    # Preview chunks
    for i, chunk in enumerate(chunks[:3]):
        print(f"\n--- Chunk {i+1} ---")
        print(f"Source: {chunk.metadata.get('source')}")
        print(f"Length: {len(chunk.page_content)}")
        print(chunk.page_content[:200])

    return chunks


# =========================================================
# STEP 3: CREATE PINECONE VECTOR STORE
# =========================================================
def create_vector_store(chunks, index_name="docs-index"):
    """
    Create a Pinecone vector store and upload document embeddings.

    Steps Performed
    ---------------
    1. Initialize OpenAI embedding model
    2. Connect to Pinecone (v3 client)
    3. Create index if it does not exist
    4. Convert text chunks → embeddings
    5. Upload embeddings to Pinecone

    Parameters
    ----------
    chunks : List[Document]
        Document chunks to embed.

    index_name : str
        Pinecone index name.

    Returns
    -------
    PineconeVectorStore
        Ready-to-use vector store for retrieval/search.
    """

    print("🔗 Connecting to Pinecone...")

    pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

    # Check if index exists
    existing_indexes = [i.name for i in pc.list_indexes()]

    if index_name not in existing_indexes:
        print(f"🆕 Creating index: {index_name}")

        pc.create_index(
            name=index_name,
            dimension=1536,  # OpenAI embedding dimension
            metric="cosine",
            spec=ServerlessSpec(
                cloud="aws",
                region="us-east-1"
            )
        )
    else:
        print(f"ℹ️ Index already exists: {index_name}")

    print("🧠 Creating embeddings...")

    embeddings = OpenAIEmbeddings(
        model="text-embedding-3-small",
        openai_api_key=os.getenv("OPENAI_API_KEY")
    )

    print("⬆️ Uploading vectors to Pinecone...")


    vectorstore = PineconeVectorStore.from_documents(
        documents=chunks,
        embedding=embeddings,
        index_name=index_name
    )

    print("✅ Vector store ready!")

    return vectorstore


# =========================================================
# MAIN PIPELINE
# =========================================================
def main():
    """
    End-to-end pipeline:
    1. Load documents
    2. Split into chunks
    3. Store in Pinecone
    """

    docs_path = "/content"  # Change to your folder

    print("\n🚀 STARTING PIPELINE\n")

    documents = load_documents(docs_path)
    chunks = split_documents(documents)
    vectorstore = create_vector_store(chunks)

    print("\n🎉 PIPELINE COMPLETED SUCCESSFULLY!")


# =========================================================
# ENTRY POINT
# =========================================================
if __name__ == "__main__":
    main()